In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, concatenate_datasets,Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold
from collections import Counter

In [ ]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}

In [ ]:
# Load all English datasets
english_datasets = [
    load_dataset("UniversalCEFR/readme_en")["train"],
    load_dataset("UniversalCEFR/cefr_asag_en")["train"],
    load_dataset("UniversalCEFR/icle500_en")["train"],
    load_dataset("UniversalCEFR/cefr_sp_en")["train"],
    load_dataset("UniversalCEFR/elg_cefr_en")["train"],
    load_dataset("UniversalCEFR/cambridge_exams_en")["train"],
]
english_data = concatenate_datasets(english_datasets)

In [ ]:
english_data

In [ ]:
# Filter to keep only valid CEFR levels
filtered_data = english_data.filter(lambda x: x["cefr_level"] in CEFR_LEVELS)

In [ ]:
# Remove duplicate texts
df = filtered_data.to_pandas().drop_duplicates(subset="text", keep="first")
filtered_data = Dataset.from_pandas(df)

In [ ]:
filtered_data

In [ ]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [ ]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [ ]:
# Tokenize the dataset
tokenized_data = filtered_data.map(preprocess, batched=True, remove_columns=filtered_data.column_names)

# Split English into train/val
n = len(tokenized_data)
train_end = int(0.8 * n)
dev_end = int(0.9 * n)

ds_train = tokenized_data.select(range(0, train_end))
ds_dev   = tokenized_data.select(range(train_end, dev_end))
ds_test  = tokenized_data.select(range(dev_end, n))

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS), trust_remote_code=True)

In [ ]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [ ]:
# Training args
args = TrainingArguments(
    output_dir="./eurobert_cefr_english_only",  
    num_train_epochs=3, 
    per_device_train_batch_size=2,              
    per_device_eval_batch_size=3,                
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_weighted_f1",
    greater_is_better=True,
    seed=42,
    learning_rate=3.6e-5,
    warmup_ratio=0.1,
    gradient_accumulation_steps=16,      
    optim="adamw_torch_fused",                   
    lr_scheduler_type="linear",                  
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    save_total_limit=1,
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,  
)

In [ ]:
# Train on English-only data
trainer.train()

# Evaluate on Dev set
trainer.evaluate()

# Save model, tokenizer, and trainer state
save_dir = "./eurobert_cefr_english_only/final_model"
trainer.save_model(save_dir)                    
tokenizer.save_pretrained(save_dir)            
trainer.state.save_to_json(os.path.join(save_dir, "trainer_state.json"))  

print(f"Model saved to {save_dir}")